In [1]:
from huggingface_hub import hf_hub_download
import importlib.util

# Descarga el loader unificado (está en cualquier repo)
loader_path = hf_hub_download("Reynier/dga-cnn", "dga_loader.py")
spec = importlib.util.spec_from_file_location("dga_loader", loader_path)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

name_model="cnn"
# Carga cualquier modelo
model, m = mod.load_dga_model(name_model)
results = mod.predict_domains(m, model, ["google.com", "xkr3f9mq.ru"])
print(results)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dga_loader.py: 0.00B [00:00, ?B/s]

dga_cnn_model_1M.pth:   0%|          | 0.00/51.9k [00:00<?, ?B/s]

model.py: 0.00B [00:00, ?B/s]

  cnn ready.
[{'domain': 'google.com', 'label': 'legit', 'score': 0.0638}, {'domain': 'xkr3f9mq.ru', 'label': 'dga', 'score': 0.9935}]


In [2]:
# Para un solo dominio, pasamos el dominio en una lista
results = mod.predict_domains(m, model, ["google.com"])

# Como devuelve una lista, accedemos al primer elemento [0]
prediccion = results[0]

print(f"Dominio: {prediccion['domain']}")
print(f"Etiqueta: {prediccion['label']}")
print(f"Puntaje: {prediccion['score']}")

Dominio: google.com
Etiqueta: legit
Puntaje: 0.0638


In [3]:
import pandas as pd
import time

families = ['bazarbackdoor.gz',
            'bazarbackdoor_v2.gz',
            'bazarbackdoor_v3.gz',
            'bigviktor.gz',
            'bumblebee.gz',
            'ccleaner.gz',
            'dmsniff.gz',
            'enviserv.gz',
            'fosniw.gz',
            'goz.gz',
            'gozi_rfc4343.gz',
            'infy.gz',
            'monerodownloader.gz',
            'newgoz.gz',
            'ngioweb.gz',
            'pandabanker.gz',
            'pizd.gz',
            'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]

runs = 30
chunk_size = 50
offset_legit = 1500

if model is None:
    print("❌ No se pudo cargar el modelo FANCI. Deteniendo evaluación.")
else:
    print("🚀 Iniciando evaluación del modelo FANCI...")

    for family in families:
        print(f"📂 Evaluando familia: {family}")

        dga = pd.read_csv(
            f'/content/drive/My Drive/New_Families/{family}',
            chunksize=chunk_size
        )

        legit = pd.read_csv(
            '/content/drive/My Drive/Familias_Test/legit.gz',
            skiprows=range(1, offset_legit + 1),
            chunksize=chunk_size
        )

        for run in range(runs):
            print(f'  🔄 {run+1:02}/{runs}', end='\r')

            try:
                dfw = pd.concat([dga.get_chunk(), legit.get_chunk()], ignore_index=True)
            except StopIteration:
                print(f"\n⚠️ No hay suficientes datos para completar {family}")
                break

            pred = []
            prob = []
            query_time = []

            for domain_to_check in dfw.domain.values:
                st = time.time()

                try:
                    results = mod.predict_domains(m, model, [domain_to_check])
                    result = results[0]

                    label_value = 1 if result['label'] == 'dga' else 0
                    probability = result['score']

                    pred.append(label_value)
                    prob.append(probability)

                except Exception:
                    pred.append(0)
                    prob.append(0.5)

                query_time.append(time.time() - st)

            dfw['pred'] = pred
            dfw['prob'] = prob
            dfw['query_time'] = query_time

            dfw.to_csv(
                f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz',
                index=False
            )

    print("\n✅ Evaluación completada!")

🚀 Iniciando evaluación del modelo FANCI...
📂 Evaluando familia: bazarbackdoor.gz
📂 Evaluando familia: bazarbackdoor_v2.gz
📂 Evaluando familia: bazarbackdoor_v3.gz
📂 Evaluando familia: bigviktor.gz
📂 Evaluando familia: bumblebee.gz
📂 Evaluando familia: ccleaner.gz
📂 Evaluando familia: dmsniff.gz
📂 Evaluando familia: enviserv.gz
📂 Evaluando familia: fosniw.gz
📂 Evaluando familia: goz.gz
📂 Evaluando familia: gozi_rfc4343.gz
📂 Evaluando familia: infy.gz
📂 Evaluando familia: monerodownloader.gz
📂 Evaluando familia: newgoz.gz
📂 Evaluando familia: ngioweb.gz
📂 Evaluando familia: pandabanker.gz
📂 Evaluando familia: pizd.gz
📂 Evaluando familia: reconyc.gz
📂 Evaluando familia: sharkbot.gz
📂 Evaluando familia: szribi.gz
📂 Evaluando familia: torpig.gz
📂 Evaluando familia: tufik.gz
📂 Evaluando familia: verblecon.gz
📂 Evaluando familia: wd.gz
📂 Evaluando familia: xshellghost.gz
  🔄 30/30
✅ Evaluación completada!


In [19]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Función para calcular FPR y TPR
def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn + 1e-10)
    tpr = tp / (tp + fn + 1e-10)
    return fpr, tpr

families = ['bazarbackdoor.gz', 'bazarbackdoor_v2.gz', 'bazarbackdoor_v3.gz', 'bigviktor.gz', 'bumblebee.gz', 'ccleaner.gz', 'dmsniff.gz', 'enviserv.gz', 'fosniw.gz', 'goz.gz', 'gozi_rfc4343.gz', 'infy.gz', 'monerodownloader.gz', 'newgoz.gz', 'ngioweb.gz', 'pandabanker.gz', 'pizd.gz', 'reconyc.gz', 'sharkbot.gz', 'szribi.gz', 'torpig.gz', 'tufik.gz', 'verblecon.gz', 'wd.gz', 'xshellghost.gz']
runs = 30

# Listas para métricas globales
all_acc, all_acc_std = [], []
all_pre, all_pre_std = [], []
all_rec, all_rec_std = [], []
all_f1, all_f1_std = [], []
all_fpr, all_fpr_std = [], []
all_tpr, all_tpr_std = [], []
all_qt, all_qts = [], []
total_unknowns_global = 0

for family in families:
    acc, pre, rec, f1, fpr, tpr, qt = [], [], [], [], [], [], []
    total_unknowns = 0

    for run in range(runs):
        path = f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz'
        if not os.path.exists(path): continue

        df = pd.read_csv(path)
        y_true = (df['label'] == 'dga').astype(int)
        y_pred = df['pred']

        acc.append(accuracy_score(y_true, y_pred))
        pre.append(precision_score(y_true, y_pred, zero_division=0))
        rec.append(recall_score(y_true, y_pred, zero_division=0))
        f1.append(f1_score(y_true, y_pred, zero_division=0))

        fpr_val, tpr_val = fpr_tpr(y_true, y_pred)
        fpr.append(fpr_val)
        tpr.append(tpr_val)
        if 'query_time' in df.columns: qt.append(df['query_time'].mean())

    if acc:
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f}')

        all_acc.append(np.mean(acc))
        all_acc_std.append(np.std(acc))
        all_pre.append(np.mean(pre))
        all_pre_std.append(np.std(pre))
        all_rec.append(np.mean(rec))
        all_rec_std.append(np.std(rec))
        all_f1.append(np.mean(f1))
        all_f1_std.append(np.std(f1))
        all_fpr.append(np.mean(fpr))
        all_fpr_std.append(np.std(fpr))
        all_tpr.append(np.mean(tpr))
        all_tpr_std.append(np.std(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.std(qt))
        total_unknowns_global += total_unknowns

print("\n### ✅ Métricas recolectadas correctamente ###")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bazarbackdoor  : acc:0.80±0.038 f1:0.76±0.054 pre:0.93±0.041 rec:0.64±0.068 FPR:0.05±0.029 TPR:0.64±0.068 QT:0.00046±0.00005
bazarbackdoor_v2: acc:0.84±0.031 f1:0.82±0.039 pre:0.94±0.036 rec:0.73±0.053 FPR:0.05±0.029 TPR:0.73±0.053 QT:0.00044±0.00009
bazarbackdoor_v3: acc:0.52±0.026 f1:0.15±0.065 pre:0.64±0.222 rec:0.09±0.039 FPR:0.05±0.029 TPR:0.09±0.039 QT:0.00032±0.00003
bigviktor      : acc:0.54±0.026 f1:0.21±0.072 pre:0.74±0.161 rec:0.12±0.048 FPR:0.05±0.029 TPR:0.12±0.048 QT:0.00033±0.00005
bumblebee      : acc:0.81±0.027 f1:0.77±0.037 pre:0.94±0.038 rec:0.66±0.052 FPR:0.05±0.029 TPR:0.66±0.052 QT:0.00035±0.00013
ccleaner       : acc:0.89±0.032 f1:0.88±0.038 pre:0.95±0.032 rec:0.82±0.060 FPR:0.05±0.029 TPR:0.82±0.060 QT:0.00041±0.00009
dmsniff        : acc:0.93±0.086 f1:0.92±0.124 pre:0.95±0.056 rec:0.91±0.161 FPR:0.05±0.029 TPR:0.91±0.161 QT:0.00050±0.

In [20]:
import pandas as pd
import numpy as np

# Crear el DataFrame usando todas las listas de medias y desviaciones
metrics_dict = {
    'Family': [f.split('.')[0] for f in families],
    'Accuracy': all_acc,
    'Acc_std': all_acc_std,
    'Precision': all_pre,
    'Pre_std': all_pre_std,
    'Recall': all_rec,
    'Rec_std': all_rec_std,
    'F1-Score': all_f1,
    'F1_std': all_f1_std,
    'FPR': all_fpr,
    'FPR_std': all_fpr_std,
    'TPR': all_tpr,
    'TPR_std': all_tpr_std,
    'Query_Time_Mean': all_qt,
    'Query_Time_Std': all_qts
}

df_metrics = pd.DataFrame(metrics_dict)

# Calcular fila de promedios globales
global_means = df_metrics.mean(numeric_only=True).to_dict()
global_means['Family'] = 'GLOBAL_MEAN'

df_metrics = pd.concat([df_metrics, pd.DataFrame([global_means])], ignore_index=True)

output_path = f'/content/drive/My Drive/results/metricas_globales_final_{name_model}.csv'
df_metrics.to_csv(output_path, index=False)

print(f"✅ CSV final con todas las desviaciones guardado en: {output_path}")
display(df_metrics)

✅ CSV final con todas las desviaciones guardado en: /content/drive/My Drive/results/metricas_globales_final_cnn.csv


,Family,Accuracy,Acc_std,Precision,Pre_std,Recall,Rec_std,F1-Score,F1_std,FPR,FPR_std,TPR,TPR_std,Query_Time_Mean,Query_Time_Std
0,bazarbackdoor,0.798333,0.037690,0.933877,0.041259,0.642667,0.068261,0.759325,0.053786,0.046,0.028821,0.642667,0.068261,0.000455,0.000051
1,bazarbackdoor_v2,0.844000,0.031048,0.942018,0.035865,0.734000,0.053454,0.823871,0.038511,0.046,0.028821,0.734000,0.053454,0.000442,0.000094
2,bazarbackdoor_v3,0.519667,0.026266,0.642672,0.221885,0.085333,0.039305,0.148975,0.065420,0.046,0.028821,0.085333,0.039305,0.000319,0.000028
3,bigviktor,0.539333,0.025552,0.735237,0.160886,0.124667,0.048079,0.209825,0.072278,0.046,0.028821,0.124667,0.048079,0.000333,0.000052
4,bumblebee,0.807000,0.027343,0.936474,0.037504,0.660000,0.051640,0.772803,0.037074,0.046,0.028821,0.660000,0.051640,0.000354,0.000130
5,ccleaner,0.885000,0.032326,0.947631,0.031512,0.816000,0.059643,0.875499,0.038055,0.046,0.028821,0.816000,0.059643,0.000406,0.000095
6,dmsniff,0.931667,0.086452,0.945665,0.055723,0.909333,0.160560,0.920937,0.124339,0.046,0.028821,0.909333,0.160560,0.000497,0.000111
7,enviserv,0.902333,0.026669,0.949517,0.030737,0.850667,0.044040,0.896625,0.029428,0.046,0.028821,0.850667,0.044040,0.000324,0.000047
8,fosniw,0.477000,0.014411,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.046,0.028821,0.000000,0.000000,0.000327,0.000047
9,goz,0.976667,0.014682,0.956711,0.026312,0.999333,0.003590,0.977374,0.014034,0.046,0.028821,0.999333,0.003590,0.000337,0.000046
